# Hardware Playground: Where Does Quantization Actually Pay Off? — coding workshop (advanced)

**Run this right after Notebook 02, in the same Colab runtime** so you keep whatever model
is already loaded there — reloading it from scratch here costs several extra minutes for
no reason. No new installs needed.

Expected time: ~40–55 minutes total, GPU needed. The notebook has three parts and they're
independent of each other in terms of what they show — if you're short on time, Part A is
the fastest and cheapest to run.

Notebook 02 compared your quantized model against Session 1's fp16 baseline, one question
at a time, and for many of you the quantized run was *not* faster or lower-energy than
fp16 — a real result, and one the lesson screen "The hardware has to support the format" predicts.
This notebook does not try to re-litigate raw speed: on a Colab T4, neither GPTQ kernel
that could make int4 genuinely faster (Marlin, then Triton) compiles successfully -- both
fail with real CUDA/C++ compiler errors specific to this GPU and this gptqmodel version,
not anything fixable from notebook code (see Step 0). The kernel that does work
(`torch`) is a correctness-only fallback, not a fair stand-in for what quantization's
speed benefit looks like on hardware where a fast kernel actually runs. What this
notebook tries instead is two angles that don't depend on which kernel is running:

- **Part A — Memory footprint.** How much GPU memory does each configuration actually use.
- **Part B — Memory headroom vs. batch size.** How much larger a batch each configuration
  can hold in GPU memory before running out -- a benefit quantization's smaller weights
  guarantee regardless of kernel speed.
- **Part C — Fit on hardware.** Whether a model too big to load in fp16 on this GPU loads
  fine once quantized.


## How this workshop works

This is the coding version of the notebook, **advanced level**. In some code cells, lines
are replaced by `# TODO` comments and `___`. Each exercise above such a cell says what the
code must do; the comments in the cell give the details. This version takes longer than
the expected time given above.

1. Replace every `___` with your code, then run the cell. Open the hints if you need them.
2. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
3. If you are stuck, the solution is below the check cell. Open it after you have tried.
   Or tick **fill_in_solutions** in the cell right below these instructions and run that cell:
   from then on, every exercise cell you run gets its solution in place of its code. With the box
   ticked, Runtime → Run all runs the whole notebook with the solutions.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

Exercises marked **(advanced)** are only in this version. In them you implement an
equation from the lesson or a paper, or a technique that makes the pipeline cheaper to
run, and check it against the library or the baseline.

In [ ]:
# @title Run this cell first. Tick the box to fill in the solutions (point 3 above).
fill_in_solutions = False  # @param {type:"boolean"}

import re

_solutions = {
    1: "def build_prompt(question, corpus):\n    context = \"\\n\\n\".join(f\"[{name}]\\n{text.strip()}\" for name, text in corpus.items())\n    messages = [\n        {\n            \"role\": \"system\",\n            \"content\": (\n                \"You are a concise assistant. Answer the question using ONLY the \"\n                \"documents provided below. Answer in one short sentence.\"\n            ),\n        },\n        {\n            \"role\": \"user\",\n            \"content\": f\"Documents:\\n{context}\\n\\nQuestion: {question}\",\n        },\n    ]\n    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n\n\nALL_PROMPTS = [build_prompt(pair[\"question\"], CORPUS) for pair in QA_PAIRS]\nprint(f\"Built {len(ALL_PROMPTS)} prompts, one per question.\")\n\n\ndef time_batched_generation(prompts, gen_model, gen_tokenizer, max_new_tokens=60):\n    inputs = gen_tokenizer(prompts, return_tensors=\"pt\", padding=True).to(gen_model.device)\n    prompt_len = inputs[\"input_ids\"].shape[1]\n\n    if torch.cuda.is_available():\n        torch.cuda.synchronize()\n    start = time.time()\n    with torch.no_grad():\n        output = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)\n    if torch.cuda.is_available():\n        torch.cuda.synchronize()\n    elapsed = time.time() - start\n\n    n_generated_tokens = output.shape[0] * (output.shape[1] - prompt_len)\n    tokens_per_sec = n_generated_tokens / elapsed\n    return elapsed, tokens_per_sec\n\n\ndef measure_peak_memory_gb(gen_model, gen_tokenizer, prompt):\n    if not torch.cuda.is_available():\n        return None\n    torch.cuda.empty_cache()\n    torch.cuda.reset_peak_memory_stats()\n    inputs = gen_tokenizer(prompt, return_tensors=\"pt\").to(gen_model.device)\n    with torch.no_grad():\n        gen_model.generate(**inputs, max_new_tokens=30, do_sample=False)\n    peak_bytes = torch.cuda.max_memory_allocated()\n    return peak_bytes / (1024 ** 3)\n\n\ndef batch_size_sweep_with_memory(prompts, gen_model, gen_tokenizer, batch_sizes, max_new_tokens=60):\n    rows = []\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    for bs in batch_sizes:\n        batch = [prompts[i % len(prompts)] for i in range(bs)]\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n            torch.cuda.reset_peak_memory_stats()\n        try:\n            time_batched_generation(batch, gen_model, gen_tokenizer, max_new_tokens=max_new_tokens)\n            peak_gb = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else float(\"nan\")\n            print(f\"batch={bs}: OK, peak GPU memory {peak_gb:.2f} GB\")\n            rows.append({\"batch_size\": bs, \"peak_memory_gb\": round(peak_gb, 2), \"oom\": False})\n        except torch.cuda.OutOfMemoryError:\n            print(f\"batch={bs}: ran out of GPU memory here -- stopping the sweep.\")\n            rows.append({\"batch_size\": bs, \"peak_memory_gb\": None, \"oom\": True})\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n            break\n    return rows",
    2: "import matplotlib.pyplot as plt\nfrom codecarbon import EmissionsTracker\n\n\ndef energy_per_token_sweep(prompts, gen_model, gen_tokenizer, batch_sizes, max_new_tokens=60):\n    rows = []\n    for bs in batch_sizes:\n        batch = [prompts[i % len(prompts)] for i in range(bs)]\n        tracker = EmissionsTracker(project_name=f\"batch-{bs}\", measure_power_secs=1,\n                                   save_to_file=False, log_level=\"error\")\n        tracker.start()\n        elapsed, tokens_per_sec = time_batched_generation(batch, gen_model, gen_tokenizer, max_new_tokens)\n        tracker.stop()\n        energy_kwh = float(tracker.final_emissions_data.energy_consumed)\n        n_tokens = tokens_per_sec * elapsed\n        joules_per_token = energy_kwh * 3.6e6 / n_tokens\n        rows.append({\"batch_size\": bs, \"tokens_per_sec\": round(tokens_per_sec, 1),\n                     \"joules_per_token\": joules_per_token})\n    return pd.DataFrame(rows)\n\n\ndef knee(sweep, min_gain=1.2):\n    sizes = list(sweep[\"batch_size\"])\n    tps = list(sweep[\"tokens_per_sec\"])\n    for i in range(len(sizes) - 1):\n        if tps[i + 1] / tps[i] < min_gain:\n            return sizes[i]\n    return sizes[-1]\n\nenergy_sweep = energy_per_token_sweep(ALL_PROMPTS, baseline_model, baseline_tokenizer, [1, 2, 4, 8, 16, 32])\nprint(energy_sweep.to_string(index=False))\nprint(f\"\\nThroughput stops growing much after batch size {knee(energy_sweep)}.\")\n\nfig, axes = plt.subplots(1, 2, figsize=(10, 3.5))\naxes[0].plot(energy_sweep[\"batch_size\"], energy_sweep[\"tokens_per_sec\"], marker=\"o\")\naxes[0].set_ylabel(\"tokens per second\")\naxes[1].plot(energy_sweep[\"batch_size\"], energy_sweep[\"joules_per_token\"], marker=\"o\")\naxes[1].set_ylabel(\"joules per token\")\nfor ax in axes:\n    ax.set_xscale(\"log\", base=2)\n    ax.set_xlabel(\"batch size\")\nplt.tight_layout()\nplt.show()",
    3: "import matplotlib.pyplot as plt\n\nfig, axes = plt.subplots(1, 2, figsize=(11, 4))\n\naxes[0].bar(memory_df[\"config\"], memory_df[\"peak_memory_gb\"], color=[\"#5a80b8\", \"#4c8c4a\"])\naxes[0].set_ylabel(\"Peak GPU memory (GB)\")\naxes[0].set_title(\"Memory footprint (batch size 1)\")\n\nok = memory_sweep_df[~memory_sweep_df[\"oom\"]]\nfor cfg, group in ok.groupby(\"config\"):\n    group = group.sort_values(\"batch_size\")\n    axes[1].plot(group[\"batch_size\"], group[\"peak_memory_gb\"], marker=\"o\", label=cfg)\naxes[1].set_xlabel(\"Batch size (questions processed together)\")\naxes[1].set_ylabel(\"Peak GPU memory (GB)\")\naxes[1].set_xscale(\"log\", base=2)\naxes[1].set_xticks(BATCH_SIZES)\naxes[1].set_xticklabels(BATCH_SIZES)\naxes[1].set_title(\"Memory headroom vs. batch size\")\naxes[1].legend()\n\nplt.tight_layout()\nplt.show()\n\nfor cfg, group in memory_sweep_df.groupby(\"config\"):\n    largest_ok = group.loc[~group[\"oom\"], \"batch_size\"].max()\n    hit_oom = bool(group[\"oom\"].any())\n    note = \"ran out of GPU memory at the next size up\" if hit_oom else \"did not run out of memory within the tested range\"\n    print(f\"{cfg}: largest batch size that fit was {largest_ok} ({note})\")\n\nmemory_df.to_csv(\"memory_results.csv\", index=False)\nmemory_sweep_df.to_csv(\"memory_sweep_results.csv\", index=False)\nprint(\"Saved memory_results.csv and memory_sweep_results.csv\")",
    4: "BIG_GPTQ_MODEL_NAME = \"Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4\"\n\nif torch.cuda.is_available():\n    torch.cuda.reset_peak_memory_stats()\n\nprint(f\"Loading {BIG_GPTQ_MODEL_NAME} (4-bit)...\")\nbig_gptq_tokenizer = AutoTokenizer.from_pretrained(BIG_GPTQ_MODEL_NAME)\nbig_gptq_model = AutoModelForCausalLM.from_pretrained(\n    BIG_GPTQ_MODEL_NAME,\n    device_map=\"auto\",\n    quantization_config=GPTQConfig(bits=4, backend=\"torch\"),\n)\n\ninputs = big_gptq_tokenizer(\"Hello, how are you?\", return_tensors=\"pt\").to(big_gptq_model.device)\nwith torch.no_grad():\n    big_gptq_model.generate(**inputs, max_new_tokens=10)\n\nprint(\"Loaded and generated successfully at 4-bit.\")\nif torch.cuda.is_available():\n    peak_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)\n    print(f\"Peak GPU memory for the 7B model at 4-bit: {peak_gb:.2f} GB\")\nprint(f\"\\nfp16 attempt succeeded: {big_model_loaded_fp16}\")\nprint(\"4-bit attempt succeeded: True\")\n\nBIG_PARAMS_BILLION = 7.6\nweights_fp16_gb = BIG_PARAMS_BILLION * 1e9 * 16 / 8 / (1024 ** 3)\nweights_int4_gb = BIG_PARAMS_BILLION * 1e9 * 4 / 8 / (1024 ** 3)\nprint(f\"Weights alone: {weights_fp16_gb:.1f} GB in fp16, {weights_int4_gb:.1f} GB in 4-bit\")",
}
_fill_in = set(_solutions) if fill_in_solutions else set()


def _fill_in_transformer(lines):
    """An exercise cell starts with `# Exercise N`; with fill-in on for N, it becomes the solution."""
    m = re.match(r"# Exercise (\d+)\s*$", lines[0]) if lines else None
    if not m or int(m.group(1)) not in _fill_in:
        return lines
    n = int(m.group(1))
    return [f"get_ipython().set_next_input(_solutions[{n}], replace=True)\n"] + (_solutions[n] + "\n").splitlines(True)


_ip = get_ipython()
_ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup
                                     if getattr(t, "__name__", "") != "_fill_in_transformer"]
_ip.input_transformers_cleanup.append(_fill_in_transformer)
if fill_in_solutions:
    print("✓ Every exercise cell you run from now on gets its solution.")

## Step 0 — reuse or reload

If `model` and `tokenizer` are still in memory from Notebook 02 in this same runtime, this
cell skips straight past the reload. If you're running this notebook on its own, it repeats
Notebook 02's GPTQ-then-bitsandbytes loading logic.


In [ ]:
!pip install -q bitsandbytes optimum codecarbon sacrebleu bert-score

import importlib.util
if importlib.util.find_spec("gptqmodel") is None:
    !pip install -q gptqmodel && echo "gptqmodel installed." || echo "gptqmodel failed to install. The GPTQ path will fall back to bitsandbytes int8."
else:
    print("gptqmodel already installed, skipping.")

**If this is the first time installing packages in this session** (i.e. you're running
this notebook on its own, not continuing right after Notebook 02): go to `Runtime` ->
`Restart session` once after the install cell above finishes, before continuing (this
keeps the packages you just installed, it only clears the running Python process).
Installing `gptqmodel` can pull in a different `numpy` build than what's
already loaded, which otherwise shows up a few cells down as a confusing `numpy.ufunc` /
`ModuleNotFoundError` when importing `transformers`. If you *are* continuing right after
Notebook 02 in the same runtime, the install cell above is a fast no-op and you can skip
the restart.


In [ ]:
import time
import gc
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GPTQConfig

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. This notebook is intended for a GPU runtime --- go to "
          "Runtime > Change runtime type, select a GPU, then Runtime > Restart session "
          "and re-run from the top before continuing.")

import importlib.util

_missing_pkgs = [p for p in ("optimum", "bitsandbytes") if importlib.util.find_spec(p) is None]
if _missing_pkgs:
    raise ImportError(
        f"{', '.join(_missing_pkgs)} not importable in this Python process, even though "
        "the install cell above should have installed them. This almost always means the "
        "runtime needs a restart after installing new packages: go to Runtime > Restart "
        "session (NOT Disconnect/Delete runtime -- that would lose the install too), then "
        "re-run this notebook from the top. You do not need to re-run the pip install "
        "cell itself, it already worked; you just need a fresh process to pick it up."
    )

if "model" not in globals() or "tokenizer" not in globals():
    print("No model found in memory --- loading from scratch (same logic as Notebook 02).")
    GPTQ_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4"
    FALLBACK_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
    try:
        tokenizer = AutoTokenizer.from_pretrained(GPTQ_MODEL_NAME)
        model = AutoModelForCausalLM.from_pretrained(
            GPTQ_MODEL_NAME,
            device_map="auto",
            quantization_config=GPTQConfig(bits=4, backend="torch"),
        )
        QUANT_CONFIG_USED = "gptq-int4"
        print(f"Loaded {GPTQ_MODEL_NAME} via GPTQ.")
    except Exception as e:
        print(f"GPTQ load failed ({type(e).__name__}: {e}).")
        print(f"Falling back to bitsandbytes 8-bit on {FALLBACK_MODEL_NAME}.")
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL_NAME)
        model = AutoModelForCausalLM.from_pretrained(
            FALLBACK_MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            dtype=torch.float16,
        )
        QUANT_CONFIG_USED = "bnb-int8"
        print(f"Loaded {FALLBACK_MODEL_NAME} via bitsandbytes int8.")
    model.eval()
else:
    print(f"Reusing the already-loaded model from this session: {QUANT_CONFIG_USED}")

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"\nQuantized config in this notebook: {QUANT_CONFIG_USED}")


## Step 1 — corpus and questions (same as Notebook 02)

Same eight documents and eight questions as Notebook 02, so the prompts built from them
here are the exact same length and difficulty as what you already benchmarked one at a
time.


In [ ]:
CORPUS = {
    "training_vs_inference": """
Training a large language model is a one-time, very expensive event: the organization
that builds the model spends a large amount of compute once. Inference — actually using
the trained model to answer a query — is comparatively cheap per request, but it happens
every single time anyone uses the model. Once a model is deployed and used by many people,
the accumulated cost of inference tends to dominate the model's total lifetime footprint,
even though any single inference looks negligible next to the training run.
Source: Patterson et al., "Carbon Emissions and Large Neural Network Training," 2021
(arXiv:2104.10350). Training GPT-3 was estimated to emit 552 metric tons of CO2-equivalent,
using about 1,287 megawatt-hours of electricity.
""",
    "data_centers": """
Data centers are the physical facilities that house the servers running most of the
world's AI models. According to the International Energy Agency, global data center
electricity consumption was about 415 terawatt-hours in 2024, roughly 1.5 percent of all
electricity consumed worldwide that year. That demand is growing fast: electricity use by
data centers rose by about 17 percent in 2025 alone, driven mostly by AI workloads, and is
projected to approach 950 terawatt-hours by 2030, or about 3 percent of global electricity
demand.
Source: International Energy Agency, "Energy and AI" and related data centre electricity
reports, 2024-2025.
""",
    "quantization_basics": """
Neural network weights are normally stored as 32-bit floating point numbers (FP32).
Quantization reduces the number of bits used to represent each weight — for example to
16-bit (FP16/BF16), 8-bit integers (INT8), or even lower. Fewer bits per number means a
smaller model file and less memory traffic when running it, at the cost of some numerical
precision. A useful analogy: quantizing a model is like reducing the number of colors used
to represent an image — the picture is still recognizable, but some fine detail is lost.
Whether this actually saves energy in practice depends on the hardware: it helps most when
the bottleneck is moving data in and out of memory, not raw computation.
Source: Grootendorst, "A Visual Guide to Quantization," 2024.
""",
    "measuring_carbon": """
Estimating the carbon footprint of running a model requires several assumptions: how much
electricity the hardware draws, the Power Usage Effectiveness (PUE) of the data center
(extra power spent on cooling and overhead), and the carbon intensity of the local
electricity grid — how much CO2 is emitted per unit of electricity, which varies a lot by
region and by time of day. Because of all these assumptions, published carbon estimates can
vary widely, and some influential early estimates have later been revised — sometimes by an
order of magnitude — once other researchers re-examined the methodology. This is one reason
this course emphasizes measuring your own numbers rather than only citing others' headline
figures.
""",
    "inference_cost_task_type": """
Not all models cost the same to run for the same task. A 2024 study compared task-specific
models (built and trained for one narrow job) against multi-purpose generative models (like
general-purpose chatbots) performing the exact same tasks. For extractive question
answering, the most efficient task-specific models emitted around 0.3 grams of
CO2-equivalent per 1,000 inferences, while general-purpose generative models emitted around
10 grams of CO2-equivalent per 1,000 inferences for the same task — roughly 33 times more.
The gap was consistent across several other tasks too, even when controlling for the number
of parameters.
Source: Luccioni, Jernite & Strubell, "Power Hungry Processing: Watts Driving the Cost of AI
Deployment?", FAccT 2024 (arXiv:2311.16863).
""",
    "rag_concept": """
Retrieval-Augmented Generation, or RAG, is a technique where a model is given relevant
external text — retrieved from a document collection — as part of its input at query time,
instead of relying only on what it "learned" during training. This lets a smaller, cheaper
model answer questions about information it was never trained on, without the cost of
retraining or fine-tuning a much larger model on that information. The retrieval step
usually uses a separate, lightweight embedding model to find which documents are relevant
before the language model ever sees them.
""",
    "small_models_trend": """
Between 2024 and 2026, there has been a strong trend toward small, efficient open-weight
language models (often called SLMs, small language models) in the 1-4 billion parameter
range, designed to run on a single consumer GPU or even a laptop CPU after quantization.
These models are trained to be very capable for their size, and combined with quantization
and retrieval, they let many real applications avoid depending on a large, general-purpose
model at all — often at a fraction of the inference cost, if the task doesn't need the
breadth of a much bigger model.
""",
    "transformer_basics": """
Modern language models are built from transformer layers. Inside each layer,
"self-attention" lets every position in the input sequence gather information from every
other position, which is what allows the model to relate words like "it" back to the noun
it refers to several words earlier. This mechanism is powerful but has a computational cost
that grows faster than the length of the input (roughly quadratically with sequence
length), and more parameters mean more numbers to multiply and move through memory for
every single token generated. This is the underlying reason why both model size and input
length are direct levers on energy cost.
""",
}

for name, text in CORPUS.items():
    print(f"--- {name} ({len(text.split())} words) ---")
print(f"\nTotal documents: {len(CORPUS)}")


In [ ]:
QA_PAIRS = [
    {
        "question": "According to Patterson et al. (2021), how many metric tons of CO2-equivalent did training GPT-3 emit?",
        "reference": ["552"],
    },
    {
        "question": "Per the IEA, roughly what percentage of global electricity consumption did data centers represent in 2024?",
        "reference": ["1.5"],
    },
    {
        "question": "What does the acronym PUE stand for?",
        "reference": ["power usage effectiveness"],
    },
    {
        "question": "What does carbon footprint estimation depend on besides hardware power draw and grid carbon intensity?",
        "reference": ["pue", "power usage effectiveness"],
    },
    {
        "question": "In the 2024 study on inference costs, about how many grams of CO2-equivalent do the most efficient task-specific models emit per 1,000 extractive question-answering inferences?",
        "reference": ["0.3"],
    },
    {
        "question": "What is the name of the technique where a model is given retrieved external text at query time instead of relying only on what it learned during training?",
        "reference": ["retrieval-augmented generation", "rag"],
    },
    {
        "question": "What size range (in billions of parameters) do small language models (SLMs) typically fall into?",
        "reference": ["1-4", "1 to 4", "1-4 billion"],
    },
    {
        "question": "What mechanism inside a transformer layer lets every position gather information from every other position?",
        "reference": ["self-attention", "attention"],
    },
]

print(f"Number of questions: {len(QA_PAIRS)}")


## Step 2 — build prompts and three measurement helpers

`time_batched_generation` runs a batch of prompts through a model, used internally below
to actually run each generation (its timing return values aren't used for
cross-configuration speed claims -- see Part B for why). `measure_peak_memory_gb` resets
CUDA's peak-memory counter, runs one generation, and reads back the highest memory the
GPU actually held during that call. `batch_size_sweep_with_memory` repeats that idea
across a range of batch sizes, doubling the batch size each step and recording peak
memory at every step until the GPU runs out of memory or the range runs out first.


### ✏️ Exercise 1 — Throughput, memory, and batches

Three gaps in the helpers:

1. `time_batched_generation`: the number of generated tokens over the whole batch, and
   tokens per second.
2. `measure_peak_memory_gb`: convert bytes to GB (1024³ bytes).
3. `batch_size_sweep_with_memory`: build a batch of `bs` prompts, cycling through the
   eight prompts when `bs` is larger than 8.

<details><summary><b>Hints</b> (click to open)</summary>

- `output` has shape (batch size, prompt length + new tokens).
- `i % n` is the remainder of `i` divided by `n`: it counts 0, 1, ..., n − 1, 0, 1, ...

</details>

In [ ]:
# Exercise 1
def build_prompt(question, corpus):
    context = "\n\n".join(f"[{name}]\n{text.strip()}" for name, text in corpus.items())
    messages = [
        {
            "role": "system",
            "content": (
                "You are a concise assistant. Answer the question using ONLY the "
                "documents provided below. Answer in one short sentence."
            ),
        },
        {
            "role": "user",
            "content": f"Documents:\n{context}\n\nQuestion: {question}",
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


ALL_PROMPTS = [build_prompt(pair["question"], CORPUS) for pair in QA_PAIRS]
print(f"Built {len(ALL_PROMPTS)} prompts, one per question.")


def time_batched_generation(prompts, gen_model, gen_tokenizer, max_new_tokens=60):
    inputs = gen_tokenizer(prompts, return_tensors="pt", padding=True).to(gen_model.device)
    prompt_len = inputs["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        output = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start

    n_generated_tokens = ___  # TODO: sequences x new tokens per sequence
    tokens_per_sec = ___
    return elapsed, tokens_per_sec


def measure_peak_memory_gb(gen_model, gen_tokenizer, prompt):
    if not torch.cuda.is_available():
        return None
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        gen_model.generate(**inputs, max_new_tokens=30, do_sample=False)
    peak_bytes = torch.cuda.max_memory_allocated()
    return ___  # TODO: in GB


def batch_size_sweep_with_memory(prompts, gen_model, gen_tokenizer, batch_sizes, max_new_tokens=60):
    rows = []
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    for bs in batch_sizes:
        batch = ___  # TODO: bs prompts, cycling through prompts
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        try:
            time_batched_generation(batch, gen_model, gen_tokenizer, max_new_tokens=max_new_tokens)
            peak_gb = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else float("nan")
            print(f"batch={bs}: OK, peak GPU memory {peak_gb:.2f} GB")
            rows.append({"batch_size": bs, "peak_memory_gb": round(peak_gb, 2), "oom": False})
        except torch.cuda.OutOfMemoryError:
            print(f"batch={bs}: ran out of GPU memory here -- stopping the sweep.")
            rows.append({"batch_size": bs, "peak_memory_gb": None, "oom": True})
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            break
    return rows


In [ ]:
# Check for Exercise 1. Run this cell as is.
_e, _tps = time_batched_generation(ALL_PROMPTS[:2], model, tokenizer, max_new_tokens=5)
assert _e > 0 and 0 < _tps * _e <= 2 * 5 + 1e-6, "2 prompts x at most 5 new tokens each"
_m = measure_peak_memory_gb(model, tokenizer, ALL_PROMPTS[0])
assert _m is None or 0.3 < _m < 16, f"{_m} GB is not a plausible peak for a 1.5B model on a T4"
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
def build_prompt(question, corpus):
    context = "\n\n".join(f"[{name}]\n{text.strip()}" for name, text in corpus.items())
    messages = [
        {
            "role": "system",
            "content": (
                "You are a concise assistant. Answer the question using ONLY the "
                "documents provided below. Answer in one short sentence."
            ),
        },
        {
            "role": "user",
            "content": f"Documents:\n{context}\n\nQuestion: {question}",
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


ALL_PROMPTS = [build_prompt(pair["question"], CORPUS) for pair in QA_PAIRS]
print(f"Built {len(ALL_PROMPTS)} prompts, one per question.")


def time_batched_generation(prompts, gen_model, gen_tokenizer, max_new_tokens=60):
    inputs = gen_tokenizer(prompts, return_tensors="pt", padding=True).to(gen_model.device)
    prompt_len = inputs["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        output = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start

    n_generated_tokens = output.shape[0] * (output.shape[1] - prompt_len)
    tokens_per_sec = n_generated_tokens / elapsed
    return elapsed, tokens_per_sec


def measure_peak_memory_gb(gen_model, gen_tokenizer, prompt):
    if not torch.cuda.is_available():
        return None
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        gen_model.generate(**inputs, max_new_tokens=30, do_sample=False)
    peak_bytes = torch.cuda.max_memory_allocated()
    return peak_bytes / (1024 ** 3)


def batch_size_sweep_with_memory(prompts, gen_model, gen_tokenizer, batch_sizes, max_new_tokens=60):
    rows = []
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    for bs in batch_sizes:
        batch = [prompts[i % len(prompts)] for i in range(bs)]
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        try:
            time_batched_generation(batch, gen_model, gen_tokenizer, max_new_tokens=max_new_tokens)
            peak_gb = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else float("nan")
            print(f"batch={bs}: OK, peak GPU memory {peak_gb:.2f} GB")
            rows.append({"batch_size": bs, "peak_memory_gb": round(peak_gb, 2), "oom": False})
        except torch.cuda.OutOfMemoryError:
            print(f"batch={bs}: ran out of GPU memory here -- stopping the sweep.")
            rows.append({"batch_size": bs, "peak_memory_gb": None, "oom": True})
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            break
    return rows
```

</details>

## Part A — memory footprint

One representative prompt (batch size 1), peak GPU memory only, isolating configuration
as the one variable — batch size is Part B's variable, not this one.


In [ ]:
memory_results = []

mem_quantized_gb = measure_peak_memory_gb(model, tokenizer, ALL_PROMPTS[0])
if mem_quantized_gb is None:
    raise RuntimeError(
        "No GPU detected (torch.cuda.is_available() is False) -- this notebook needs a "
        "GPU runtime for every measurement below. If you had a GPU earlier in this "
        "session, it likely disconnected: check Runtime > Change runtime type, then "
        "Runtime > Restart session and re-run from the top."
    )
memory_results.append({"config": QUANT_CONFIG_USED, "peak_memory_gb": round(mem_quantized_gb, 2)})
print(f"Peak GPU memory, {QUANT_CONFIG_USED}: {mem_quantized_gb:.2f} GB")


## Part B — memory headroom vs. batch size, this configuration

Batch sizes double from 1 up to 128, cycling back through the same eight questions once
the batch size passes eight -- repeating a prompt doesn't change how much memory it
costs, only its content would, and content isn't what this part measures.

This notebook does not compare tokens-per-second across configurations. Step 0 already
explains why: on this GPU, neither Marlin nor Triton (the two GPTQ kernels that could
make int4 genuinely faster) compiles successfully, so the only kernel that actually runs
(`torch`) is a correctness-only fallback -- not representative of quantized inference
speed on hardware where a fast kernel is available. Comparing its speed against fp16
would measure "how slow is the fallback kernel," not "is quantization faster."

What this sweep measures instead -- peak GPU memory at each batch size -- doesn't depend
on which kernel is running. Quantized weights take less memory at every batch size,
which means more requests fit in the same GPU before running out. That's a benefit
quantization's smaller weights guarantee, independent of kernel speed.


In [ ]:
BATCH_SIZES = [1, 2, 4, 8, 16, 32, 64, 128]
memory_sweep_results = []

print(f"Sweeping batch sizes for {QUANT_CONFIG_USED}...")
sweep_rows = batch_size_sweep_with_memory(ALL_PROMPTS, model, tokenizer, BATCH_SIZES)
for row in sweep_rows:
    row["config"] = QUANT_CONFIG_USED
memory_sweep_results.extend(sweep_rows)


## Step 3 — free this model, load the fp16 baseline

To compare against something, free the quantized model's GPU memory first, then load the
plain fp16 model and repeat both measurements.


In [ ]:
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

BASELINE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
baseline_tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL_NAME)
baseline_tokenizer.padding_side = "left"
if baseline_tokenizer.pad_token is None:
    baseline_tokenizer.pad_token = baseline_tokenizer.eos_token

baseline_model = AutoModelForCausalLM.from_pretrained(
    BASELINE_MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
)
baseline_model.eval()
print(f"Loaded {BASELINE_MODEL_NAME} in fp16.")


In [ ]:
mem_baseline_gb = measure_peak_memory_gb(baseline_model, baseline_tokenizer, ALL_PROMPTS[0])
if mem_baseline_gb is None:
    raise RuntimeError(
        "No GPU detected (torch.cuda.is_available() is False) -- this notebook needs a "
        "GPU runtime for every measurement below. If you had a GPU earlier in this "
        "session, it likely disconnected: check Runtime > Change runtime type, then "
        "Runtime > Restart session and re-run from the top."
    )
memory_results.append({"config": "baseline-fp16", "peak_memory_gb": round(mem_baseline_gb, 2)})
print(f"Peak GPU memory, baseline-fp16: {mem_baseline_gb:.2f} GB")

memory_df = pd.DataFrame(memory_results)
memory_df


In [ ]:
print("Sweeping batch sizes for baseline-fp16...")
sweep_rows = batch_size_sweep_with_memory(ALL_PROMPTS, baseline_model, baseline_tokenizer, BATCH_SIZES)
for row in sweep_rows:
    row["config"] = "baseline-fp16"
memory_sweep_results.extend(sweep_rows)

memory_sweep_df = pd.DataFrame(memory_sweep_results)
memory_sweep_df


### ✏️ Exercise 2 — Energy per token, and where batching stops paying off (advanced)

Generating one token for one sequence reads all the weights from GPU memory. A batch
of sequences shares that read, so larger batches usually produce more tokens per second
and use less energy per token, until the GPU's arithmetic becomes the limit.

Complete the two functions:

1. `energy_per_token_sweep` measures each batch size with CodeCarbon and computes
   **joules per generated token**. 1 kWh = 3,600,000 J. The number of tokens is
   tokens per second × seconds.
2. `knee` returns the first batch size after which doubling the batch raises the
   throughput by less than `min_gain` (by default 1.2, that is, by less than 20%). If that
   never happens, it returns the largest batch size.

The cell measures the fp16 baseline model, which is still loaded. Short runs give rough
energy numbers, so expect some noise.

**Reflection:** a service that answers one user at a time runs at batch size 1. What does
your table say about the energy cost of that, compared with collecting requests into
batches? What does a user give up when requests are batched?

<details><summary><b>Hints</b> (click to open)</summary>

- `tokens_per_sec * elapsed` is the number of generated tokens.
- Joules per token: `energy_kwh * 3.6e6 / n_tokens`.
- In `knee`, compare `tps[i + 1] / tps[i]` with `min_gain` for each consecutive pair.

</details>

In [ ]:
# Exercise 2
import matplotlib.pyplot as plt
from codecarbon import EmissionsTracker


def energy_per_token_sweep(prompts, gen_model, gen_tokenizer, batch_sizes, max_new_tokens=60):
    rows = []
    for bs in batch_sizes:
        batch = [prompts[i % len(prompts)] for i in range(bs)]
        tracker = EmissionsTracker(project_name=f"batch-{bs}", measure_power_secs=1,
                                   save_to_file=False, log_level="error")
        tracker.start()
        elapsed, tokens_per_sec = time_batched_generation(batch, gen_model, gen_tokenizer, max_new_tokens)
        tracker.stop()
        energy_kwh = float(tracker.final_emissions_data.energy_consumed)
        n_tokens = ___  # TODO: generated tokens in this batch
        joules_per_token = ___  # TODO: energy per token, in joules
        rows.append({"batch_size": bs, "tokens_per_sec": round(tokens_per_sec, 1),
                     "joules_per_token": joules_per_token})
    return pd.DataFrame(rows)


def knee(sweep, min_gain=1.2):
    sizes = list(sweep["batch_size"])
    tps = list(sweep["tokens_per_sec"])
    # TODO: the first batch size whose next doubling gains less than min_gain x throughput;
    # the largest batch size if every doubling gains at least that much
    ___

energy_sweep = energy_per_token_sweep(ALL_PROMPTS, baseline_model, baseline_tokenizer, [1, 2, 4, 8, 16, 32])
print(energy_sweep.to_string(index=False))
print(f"\nThroughput stops growing much after batch size {knee(energy_sweep)}.")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(energy_sweep["batch_size"], energy_sweep["tokens_per_sec"], marker="o")
axes[0].set_ylabel("tokens per second")
axes[1].plot(energy_sweep["batch_size"], energy_sweep["joules_per_token"], marker="o")
axes[1].set_ylabel("joules per token")
for ax in axes:
    ax.set_xscale("log", base=2)
    ax.set_xlabel("batch size")
plt.tight_layout()
plt.show()

In [ ]:
# Check for Exercise 2. Run this cell as is.
import pandas as _pd
_fake = _pd.DataFrame({"batch_size": [1, 2, 4, 8, 16], "tokens_per_sec": [10, 19, 36, 40, 41]})
assert knee(_fake) == 4, "36 -> 40 is the first doubling that gains less than 1.2x"
assert knee(_pd.DataFrame({"batch_size": [1, 2], "tokens_per_sec": [10, 30]})) == 2
assert list(energy_sweep["batch_size"]) == [1, 2, 4, 8, 16, 32]
assert (energy_sweep["joules_per_token"] > 0).all()
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
import matplotlib.pyplot as plt
from codecarbon import EmissionsTracker


def energy_per_token_sweep(prompts, gen_model, gen_tokenizer, batch_sizes, max_new_tokens=60):
    rows = []
    for bs in batch_sizes:
        batch = [prompts[i % len(prompts)] for i in range(bs)]
        tracker = EmissionsTracker(project_name=f"batch-{bs}", measure_power_secs=1,
                                   save_to_file=False, log_level="error")
        tracker.start()
        elapsed, tokens_per_sec = time_batched_generation(batch, gen_model, gen_tokenizer, max_new_tokens)
        tracker.stop()
        energy_kwh = float(tracker.final_emissions_data.energy_consumed)
        n_tokens = tokens_per_sec * elapsed
        joules_per_token = energy_kwh * 3.6e6 / n_tokens
        rows.append({"batch_size": bs, "tokens_per_sec": round(tokens_per_sec, 1),
                     "joules_per_token": joules_per_token})
    return pd.DataFrame(rows)


def knee(sweep, min_gain=1.2):
    sizes = list(sweep["batch_size"])
    tps = list(sweep["tokens_per_sec"])
    for i in range(len(sizes) - 1):
        if tps[i + 1] / tps[i] < min_gain:
            return sizes[i]
    return sizes[-1]

energy_sweep = energy_per_token_sweep(ALL_PROMPTS, baseline_model, baseline_tokenizer, [1, 2, 4, 8, 16, 32])
print(energy_sweep.to_string(index=False))
print(f"\nThroughput stops growing much after batch size {knee(energy_sweep)}.")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(energy_sweep["batch_size"], energy_sweep["tokens_per_sec"], marker="o")
axes[0].set_ylabel("tokens per second")
axes[1].plot(energy_sweep["batch_size"], energy_sweep["joules_per_token"], marker="o")
axes[1].set_ylabel("joules per token")
for ax in axes:
    ax.set_xscale("log", base=2)
    ax.set_xlabel("batch size")
plt.tight_layout()
plt.show()
```

</details>

## Step 4 — plot both results


### ✏️ Exercise 3 — Largest batch that fit

For each configuration, find the largest batch size that ran **without** running
out of memory.

<details><summary><b>Hints</b> (click to open)</summary>

- `~` negates a column of booleans: `~group["oom"]` is True where `oom` is False.

</details>

In [ ]:
# Exercise 3
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(memory_df["config"], memory_df["peak_memory_gb"], color=["#5a80b8", "#4c8c4a"])
axes[0].set_ylabel("Peak GPU memory (GB)")
axes[0].set_title("Memory footprint (batch size 1)")

ok = memory_sweep_df[~memory_sweep_df["oom"]]
for cfg, group in ok.groupby("config"):
    group = group.sort_values("batch_size")
    axes[1].plot(group["batch_size"], group["peak_memory_gb"], marker="o", label=cfg)
axes[1].set_xlabel("Batch size (questions processed together)")
axes[1].set_ylabel("Peak GPU memory (GB)")
axes[1].set_xscale("log", base=2)
axes[1].set_xticks(BATCH_SIZES)
axes[1].set_xticklabels(BATCH_SIZES)
axes[1].set_title("Memory headroom vs. batch size")
axes[1].legend()

plt.tight_layout()
plt.show()

for cfg, group in memory_sweep_df.groupby("config"):
    largest_ok = ___  # TODO: largest batch_size in group that did not run out of memory
    hit_oom = bool(group["oom"].any())
    note = "ran out of GPU memory at the next size up" if hit_oom else "did not run out of memory within the tested range"
    print(f"{cfg}: largest batch size that fit was {largest_ok} ({note})")

memory_df.to_csv("memory_results.csv", index=False)
memory_sweep_df.to_csv("memory_sweep_results.csv", index=False)
print("Saved memory_results.csv and memory_sweep_results.csv")


In [ ]:
# Check for Exercise 3. Run this cell as is.
for _cfg, _g in memory_sweep_df.groupby("config"):
    _ok = _g[_g["oom"] == False]["batch_size"]
    assert _ok.max() == _g.loc[~_g["oom"], "batch_size"].max()
assert largest_ok == group[group["oom"] == False]["batch_size"].max()
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(memory_df["config"], memory_df["peak_memory_gb"], color=["#5a80b8", "#4c8c4a"])
axes[0].set_ylabel("Peak GPU memory (GB)")
axes[0].set_title("Memory footprint (batch size 1)")

ok = memory_sweep_df[~memory_sweep_df["oom"]]
for cfg, group in ok.groupby("config"):
    group = group.sort_values("batch_size")
    axes[1].plot(group["batch_size"], group["peak_memory_gb"], marker="o", label=cfg)
axes[1].set_xlabel("Batch size (questions processed together)")
axes[1].set_ylabel("Peak GPU memory (GB)")
axes[1].set_xscale("log", base=2)
axes[1].set_xticks(BATCH_SIZES)
axes[1].set_xticklabels(BATCH_SIZES)
axes[1].set_title("Memory headroom vs. batch size")
axes[1].legend()

plt.tight_layout()
plt.show()

for cfg, group in memory_sweep_df.groupby("config"):
    largest_ok = group.loc[~group["oom"], "batch_size"].max()
    hit_oom = bool(group["oom"].any())
    note = "ran out of GPU memory at the next size up" if hit_oom else "did not run out of memory within the tested range"
    print(f"{cfg}: largest batch size that fit was {largest_ok} ({note})")

memory_df.to_csv("memory_results.csv", index=False)
memory_sweep_df.to_csv("memory_sweep_results.csv", index=False)
print("Saved memory_results.csv and memory_sweep_results.csv")
```

</details>

**Reflection:** Part A should show a clear win for the quantized configuration on memory
--- does it, in your `memory_df`? Part B extends that across batch sizes: look at your
own `memory_sweep_df`. At every batch size both configurations managed to run, is the
quantized configuration's peak memory lower than fp16's? Did either configuration hit its
memory ceiling (check the `oom` column) before reaching batch size 128, and if so, which
one, and at what batch size? What does that tell you about how many requests each
configuration could realistically serve at once on this GPU?

If *both* configurations ran out of memory at the same batch size, think about why:
quantization shrinks the model's stored weights, but not the KV cache -- the per-token
memory each generated token adds while the model runs, which grows with batch size no
matter how the weights are stored (see "Choosing the range is called calibration" in
the lesson). Could that KV cache end up mattering more than the weight savings once the
batch size is large enough?


<details><summary><b>Possible answers</b> (open it after you have thought about it)</summary>

Yes. In our run (11 GB GPU, bitsandbytes int8 because GPTQ did not load), peak memory at batch size 1 was about 1.8 GB for int8 and 3.0 GB for fp16. int8 stayed below fp16 at every batch size that ran, but the gap shrank from about 1.2 GB to 0.4 GB by batch size 32. Both ran out of memory at the same step, batch size 64, so each could serve at most 32 requests at once: quantization bought no extra room.

At large batches most memory is not the weights. Each prompt is about 1,350 tokens (all eight documents plus the question). For 32 prompts the KV cache alone takes over 1 GB, and the temporary activations for processing the prompts take more: the allocation that failed at batch size 64 was a single 1.5 GB (fp16) to 3 GB (int8) buffer. Quantizing the weights shrinks neither. In our run int8's memory even grew a little faster with the batch (about 4.6 GB from batch size 1 to 32, against 3.8 GB for fp16). So yes: past some batch size, this per-request memory outweighs the one-time weight savings.

A 16 GB T4 has about 5 GB more room, so the out-of-memory point may come one step later there. Your own `oom` column shows where it falls; expect both configurations to stop at the same or nearly the same batch size.

</details>

## Part C — fit on hardware

The lesson screen "Which models fit on a 16 GB GPU?" worked this out for a 7B model.
Here it's an actual attempted load: a ~7B model in fp16 on this GPU, then the same model
quantized to 4-bit.

**Before running the next cell:** if it leaves the GPU in a bad state (later cells all
fail with CUDA errors even though this cell itself finished), Runtime → Restart session
and continue from the cell after next — the demonstration already made its point in the
printed output.


In [ ]:
del baseline_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

BIG_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print(f"Attempting to load {BIG_MODEL_NAME} in fp16 on this GPU...")
big_model_loaded_fp16 = False
try:
    big_tokenizer = AutoTokenizer.from_pretrained(BIG_MODEL_NAME)
    big_model = AutoModelForCausalLM.from_pretrained(
        BIG_MODEL_NAME,
        dtype=torch.float16,
        device_map="auto",
    )
    inputs = big_tokenizer("Hello, how are you?", return_tensors="pt").to(big_model.device)
    with torch.no_grad():
        big_model.generate(**inputs, max_new_tokens=10)
    print("Loaded and generated fine --- if your runtime has more than 16 GB of GPU "
          "memory, this may just work; the T4's 16 GB is the case the lesson assumes.")
    big_model_loaded_fp16 = True
except Exception as e:
    print(f"Failed to load or run in fp16 ({type(e).__name__}): {e}")
    print("This is the expected outcome on a 16 GB T4 --- a ~7B model needs about 14 GB "
          "for weights alone in fp16, leaving too little room for activations.")
    if "big_model" in globals():
        del big_model
    gc.collect()
    torch.cuda.empty_cache()


### ✏️ Exercise 4 — Predict the 7B memory, then compare

Qwen2.5-7B has about 7.6 billion parameters. Compute the memory for its weights
alone at 16 bits and at 4 bits, in GB (1024³ bytes), and compare with the peak memory
printed above.

<details><summary><b>Hints</b> (click to open)</summary>

- This is the formula from the Memory Estimates Playground: parameters × bits ÷ 8 bytes.

</details>

In [ ]:
# Exercise 4
BIG_GPTQ_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

print(f"Loading {BIG_GPTQ_MODEL_NAME} (4-bit)...")
big_gptq_tokenizer = AutoTokenizer.from_pretrained(BIG_GPTQ_MODEL_NAME)
big_gptq_model = AutoModelForCausalLM.from_pretrained(
    BIG_GPTQ_MODEL_NAME,
    device_map="auto",
    quantization_config=GPTQConfig(bits=4, backend="torch"),
)

inputs = big_gptq_tokenizer("Hello, how are you?", return_tensors="pt").to(big_gptq_model.device)
with torch.no_grad():
    big_gptq_model.generate(**inputs, max_new_tokens=10)

print("Loaded and generated successfully at 4-bit.")
if torch.cuda.is_available():
    peak_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    print(f"Peak GPU memory for the 7B model at 4-bit: {peak_gb:.2f} GB")
print(f"\nfp16 attempt succeeded: {big_model_loaded_fp16}")
print("4-bit attempt succeeded: True")

BIG_PARAMS_BILLION = 7.6
weights_fp16_gb = ___  # TODO: weight memory at 16 bits, in GB
weights_int4_gb = ___  # TODO: weight memory at 4 bits, in GB
print(f"Weights alone: {weights_fp16_gb:.1f} GB in fp16, {weights_int4_gb:.1f} GB in 4-bit")


In [ ]:
# Check for Exercise 4. Run this cell as is.
assert abs(weights_fp16_gb - 14.16) < 0.05, weights_fp16_gb
assert abs(weights_int4_gb - 3.54) < 0.05, weights_int4_gb
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
BIG_GPTQ_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

print(f"Loading {BIG_GPTQ_MODEL_NAME} (4-bit)...")
big_gptq_tokenizer = AutoTokenizer.from_pretrained(BIG_GPTQ_MODEL_NAME)
big_gptq_model = AutoModelForCausalLM.from_pretrained(
    BIG_GPTQ_MODEL_NAME,
    device_map="auto",
    quantization_config=GPTQConfig(bits=4, backend="torch"),
)

inputs = big_gptq_tokenizer("Hello, how are you?", return_tensors="pt").to(big_gptq_model.device)
with torch.no_grad():
    big_gptq_model.generate(**inputs, max_new_tokens=10)

print("Loaded and generated successfully at 4-bit.")
if torch.cuda.is_available():
    peak_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    print(f"Peak GPU memory for the 7B model at 4-bit: {peak_gb:.2f} GB")
print(f"\nfp16 attempt succeeded: {big_model_loaded_fp16}")
print("4-bit attempt succeeded: True")

BIG_PARAMS_BILLION = 7.6
weights_fp16_gb = BIG_PARAMS_BILLION * 1e9 * 16 / 8 / (1024 ** 3)
weights_int4_gb = BIG_PARAMS_BILLION * 1e9 * 4 / 8 / (1024 ** 3)
print(f"Weights alone: {weights_fp16_gb:.1f} GB in fp16, {weights_int4_gb:.1f} GB in 4-bit")
```

</details>

**Reflection:** How is what Part C shows different in kind from what Parts A and B
showed? Why does a 1.5B model, like the rest of this notebook uses, never hit the kind of
wall Part C demonstrates, on this same GPU?


<details><summary><b>Possible answers</b> (open it after you have thought about it)</summary>

Parts A and B are about degree: the 1.5B model runs in both formats, and quantization changes how much memory it uses and how large a batch fits. Part C is about whether a model runs on the GPU at all. Qwen2.5-7B needs about 14–15 GB for its weights alone in fp16 (7.6 billion parameters × 2 bytes), more than our 11 GB card and nearly all of a T4's 16 GB; at 4 bits, about 4 GB.

Read the fp16 cell's output carefully. In our run it said "Loaded and generated fine", because `device_map="auto"` quietly moved about half of the layers to the computer's ordinary (CPU) memory. Generating 10 tokens then took about 12 seconds; forcing the whole model onto the GPU ran out of memory. On a T4 expect something similar: a few layers moved to the CPU, or a failed load. The 4-bit GPTQ cell could not run here (gptqmodel was not installed); a bitsandbytes 4-bit version of the same model, as a stand-in, peaked at about 5.6 GB and answered in half a second.

A 1.5B model needs only about 3 GB in fp16, so it fits with plenty of room in either format; only large batches fill the GPU, which is Part B's limit.

</details>

## Step 5 — share-out (discussion)

Compare notes with your neighbor before wrapping up.

- Which configuration loaded for you in Step 0 — GPTQ or the bitsandbytes fallback? Did
  that change what Part B's throughput curve looked like?
- In Part B, did the quantized configuration ever overtake fp16 in your results? At what
  batch size, if any?
- In Part C, did the fp16 7B load fail the same way for everyone, or did some of you get a
  bigger GPU allocation than others this session?


## Code literacy check

Before you're done, pick at least two or three code cells above that you actually ran and
read today and add `#` comments to them explaining, in your own words, what each meaningful
line or block does and *why* it's there, in language a classmate who hasn't seen the code
could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.
